# PROJECT STAGE

Current stage: Research exploration

Code may later be migrated to src modules once stabilized.

# 1. Metadata

In [ ]:
# ============================================================
# NOTEBOOK METADATA
# ============================================================

NOTEBOOK_NAME = "03_systemic_market_characterization"
AUTHOR = "Juan Manuel Giacame"
CREATED = "2026-03-29"
UPDATED = "2026-03-29"
GIT_HASH = "..."

# 2. Imports


In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
from quant_research.data.registry.universe_registry import Asset, get_all_assets
from quant_research.panels.builders.panel_builder import PanelBuilder
from quant_research.features.loaders.asset_feature_loader import FeatureLoader
from quant_research.data.processed.loaders.asset_processed_loader import AssetProcessedDataLoader
from quant_research.config.paths import SYSTEMIC_PATH 

# 3. CONFIGURATION


In [ ]:
assets = [a.symbol for a in get_all_assets()]
output_path = SYSTEMIC_PATH

panel_aligment = "union"  # intersection | union

panel_nan_policy = "keep" # keep | ffill | bfill | drop | 

config = {
    "name": "baseline",

    "assets": assets,

    "features": {
        "momentum": {
            "source_feature": "MOM_63_Z",
            "lookback": 63,
            "transform": "zscore"
        },
        "volatility": {
            "source_feature": "VOL_63_Z",
            "lookback": 63,
            "transform": "zscore"
        }
    },

    "nan_handling": {
    "warmup": {
        "method": "cut",          # cut | keep
        "based_on": "used_features"  # used_features | all
    },
    "calendar": {
        "method": "ffill",        # ffill | bfill | drop | none
    },
    "final": {
        "method": "dropna"        # dropna | keep
    }
    },

    "systemic_features": [
        "mean_mom",
        "mean_vol",
        "dispersion",
        "mom_disp",
        "breadth_mom",
        "ret_corr",
        "mom_corr"
    ],

    "parameters": {
        "correlation_window": 20,
        "smoothing_window": 20
    }
}

In [ ]:
print(assets)

# 4. Data Loading

In [ ]:
feature_loader = FeatureLoader()
processed_loader = AssetProcessedDataLoader()

builder = PanelBuilder(feature_loader, processed_loader)
# symbols = ["BTC", "SPY", "QQQ", "XLE"]

panel = builder.build_panel(
    assets=assets,
    families=["momentum", "volatility"],
    include_processed=["RET_1", "PRICE"],
    alignment=panel_aligment,
    nan_policy=panel_nan_policy,
)

## 4.1 Panel data audit

In [ ]:
import pandas as pd
import numpy as np

def audit_panel(panel: pd.DataFrame):
    
    assert isinstance(panel.columns, pd.MultiIndex), "Panel must have MultiIndex columns"

    features = panel.columns.get_level_values(0).unique()
    assets = panel.columns.get_level_values(1).unique()

    report = {}

    # ============================================================
    # SUMMARY
    # ============================================================

    report["summary"] = {
        "shape": panel.shape,
        "n_features": len(features),
        "n_assets": len(assets),
        "start": str(panel.index.min()),
        "end": str(panel.index.max()),
        "n_dates": panel.index.nunique(),
    }

    # ============================================================
    # NaNs (GLOBAL)
    # ============================================================

    total_nans = panel.isna().sum().sum()
    total_values = panel.size

    report["summary"]["nan_ratio"] = float(total_nans / total_values)

    # ============================================================
    # PER FEATURE
    # ============================================================

    per_feature = []

    for feat in features:
        df_feat = panel[feat]

        nan_ratio = df_feat.isna().sum().sum() / df_feat.size

        per_feature.append({
            "feature": feat,
            "nan_ratio": float(nan_ratio),
            "mean": float(df_feat.mean().mean()),
            "std": float(df_feat.std().mean()),
            "min": float(df_feat.min().min()),
            "max": float(df_feat.max().max()),
        })

    report["per_feature"] = pd.DataFrame(per_feature)

    # ============================================================
    # PER ASSET
    # ============================================================

    per_asset = []

    for asset in assets:
        df_asset = panel.xs(asset, level=1, axis=1)

        nan_ratio = df_asset.isna().sum().sum() / df_asset.size

        per_asset.append({
            "asset": asset,
            "nan_ratio": float(nan_ratio),
            "mean": float(df_asset.mean().mean()),
            "std": float(df_asset.std().mean()),
        })

    report["per_asset"] = pd.DataFrame(per_asset)

    # ============================================================
    # COVERAGE CHECK
    # ============================================================

    coverage = panel.notna().T.groupby(level="feature").mean().T

    report["coverage"] = coverage

    # ============================================================
    # OUTLIER CHECK (simple z-score)
    # ============================================================

    z = (panel - panel.mean()) / panel.std()
    extreme = (np.abs(z) > 5).sum().sum()

    report["summary"]["extreme_values"] = int(extreme)

    return report

In [ ]:
audit = audit_panel(panel)
audit["summary"]

In [ ]:
audit["per_feature"].sort_values("nan_ratio", ascending=False)

In [ ]:
audit["per_asset"].sort_values("nan_ratio", ascending=False)

In [ ]:
audit["coverage"].tail()

# 5. Core Computations

## 5.1 NaNs Handling

In [ ]:
def prepare_panel_for_systemic(panel: pd.DataFrame, config: dict):
    
    nan_cfg = config["nan_handling"]

    # ============================================================
    # 1. SELECT FEATURES USED
    # ============================================================

    mom_feature = config["features"]["momentum"]["source_feature"]
    vol_feature = config["features"]["volatility"]["source_feature"]

    required_features = [mom_feature, vol_feature, "RET_1"]

    if nan_cfg["warmup"]["based_on"] == "used_features":
        panel_work = panel[required_features].copy()
    else:
        panel_work = panel.copy()

    # ============================================================
    # 2. WARMUP HANDLING
    # ============================================================

    warmup_start = None

    if nan_cfg["warmup"]["method"] == "cut":
        first_valid = panel_work.apply(lambda col: col.first_valid_index())
        warmup_start = max(first_valid)
        panel_work = panel_work[panel_work.index >= warmup_start]

    # ============================================================
    # 3. CALENDAR HANDLING
    # ============================================================
    
    calendar_method = nan_cfg["calendar"]["method"]
    
    if calendar_method == "ffill":
        panel_work = panel_work.ffill()
    
    elif calendar_method == "bfill":
        panel_work = panel_work.bfill()
    
    elif calendar_method == "drop":
        # elimina filas donde hay cualquier NaN
        panel_work = panel_work.dropna()
    
    elif calendar_method == "none":
        pass
    
    else:
        raise ValueError("Invalid calendar NaN method")
    # ============================================================
    # 4. RETURN
    # ============================================================

    return panel_work, {
        "warmup_start": str(warmup_start) if warmup_start else None,
        "features_used": required_features
    }

In [ ]:
panel_prepared, nan_info = prepare_panel_for_systemic(panel, config)

## 5.2 Compute Systemic Features

In [ ]:
mom_feature = config["features"]["momentum"]["source_feature"]
vol_feature = config["features"]["volatility"]["source_feature"]

In [ ]:
mom = panel_prepared[mom_feature].copy()
vol = panel_prepared[vol_feature].copy()

returns = panel_prepared["RET_1"].copy()

In [ ]:
mean_mom = mom.mean(axis=1)
mean_vol = vol.mean(axis=1)

In [ ]:
dispersion = returns.std(axis=1)
mom_disp = mom.std(axis=1)

In [ ]:
breadth_mom = (mom > 0).sum(axis=1) / mom.shape[1]

In [ ]:
window = config["parameters"]["correlation_window"]

rolling_corr = returns.rolling(window).corr()

In [ ]:
import numpy as np

def mean_correlation(corr_matrix):
    mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
    return corr_matrix.where(mask).stack().mean()

ret_corr = rolling_corr.groupby(level=0).apply(mean_correlation)

In [ ]:
rolling_corr_mom = mom.rolling(window).corr()
mom_corr = rolling_corr_mom.groupby(level=0).apply(mean_correlation)

In [ ]:
smooth = config["parameters"]["smoothing_window"]

ret_corr = ret_corr.rolling(smooth).mean()
mom_corr = mom_corr.rolling(smooth).mean()

In [ ]:
df_systemic = pd.concat([
    mean_mom.rename("mean_mom"),
    mean_vol.rename("mean_vol"),
    dispersion.rename("dispersion"),
    mom_disp.rename("mom_disp"),
    breadth_mom.rename("breadth_mom"),
    ret_corr.rename("ret_corr"),
    mom_corr.rename("mom_corr"),
], axis=1)

# df_systemic = df_systemic.dropna()

In [ ]:
if config["nan_handling"]["final"]["method"] == "dropna":
    df_systemic = df_systemic.dropna()

In [ ]:
df_systemic_z = (df_systemic - df_systemic.mean()) / df_systemic.std()

# 6. Systemic Dataset Audit

In [ ]:
import pandas as pd
import numpy as np

def audit_systemic(df: pd.DataFrame):
    
    report = {}

    # ============================================================
    # BASIC STRUCTURE
    # ============================================================

    assert isinstance(df, pd.DataFrame), "Systemic must be DataFrame"
    assert not df.empty, "Systemic DataFrame is empty"
    assert isinstance(df.index, pd.DatetimeIndex), "Index must be DatetimeIndex"
    assert df.index.is_monotonic_increasing, "Index must be sorted"

    report["shape"] = df.shape
    report["start"] = str(df.index.min())
    report["end"] = str(df.index.max())

    # ============================================================
    # NaNs (STRICT)
    # ============================================================

    total_nans = df.isna().sum().sum()
    assert total_nans == 0, f"Systemic contains NaNs: {total_nans}"

    report["nan_check"] = "passed"

    # ============================================================
    # DUPLICATES
    # ============================================================

    assert not df.index.duplicated().any(), "Duplicate timestamps found"
    report["duplicates"] = "none"

    # ============================================================
    # CONSTANT FEATURES
    # ============================================================

    constant_cols = [col for col in df.columns if df[col].std() == 0]

    assert len(constant_cols) == 0, f"Constant features found: {constant_cols}"

    report["constant_features"] = "none"

    # ============================================================
    # EXTREME VALUES
    # ============================================================

    z = (df - df.mean()) / df.std()
    extreme = (np.abs(z) > 8).sum().sum()

    report["extreme_values"] = int(extreme)

    # no assert → solo warning
    if extreme > 0:
        print(f"⚠️ Warning: {extreme} extreme values detected")

    # ============================================================
    # VALUE RANGES (sanity)
    # ============================================================

    ranges = {}

    for col in df.columns:
        ranges[col] = {
            "min": float(df[col].min()),
            "max": float(df[col].max())
        }

    report["ranges"] = ranges

    # ============================================================
    # FINAL STATUS
    # ============================================================

    report["status"] = "PASSED"

    return report

In [ ]:
audit_sys = audit_systemic(df_systemic)
audit_sys

# 7. Export

In [ ]:
import pandas as pd

metadata = {
    "name": config["name"],
    "created_at": pd.Timestamp.now().isoformat(),

    "config": config,

    "data": {
        "n_rows": df_systemic.shape[0],
        "n_cols": df_systemic.shape[1],
        "columns": list(df_systemic.columns)
    },

    "date_range": {
        "start": str(df_systemic.index.min()),
        "end": str(df_systemic.index.max())
    },


    "audit": {
        "panel_nan_ratio": audit["summary"]["nan_ratio"],
        "systemic_status": audit_sys["status"],
        "extreme_values": audit_sys["extreme_values"]
    }
}

In [ ]:
metadata["panel_audit"] = {
    "nan_ratio": audit["summary"]["nan_ratio"],
    "extreme_values": audit["summary"]["extreme_values"],
}

In [ ]:
metadata["systemic_audit"] = {
    "status": audit_sys["status"],
    "extreme_values": audit_sys["extreme_values"],
    "shape": audit_sys["shape"]
}

In [ ]:
metadata["nan_handling"] = {
    **config["nan_handling"],
    "resolved": nan_info
}

In [ ]:
metadata["alignment"] = {
    "method": panel_aligment,
    "reason": "preserve full calendar for systemic analysis"
}

In [ ]:
import hashlib

hash_val = hashlib.md5(pd.util.hash_pandas_object(df_systemic).values).hexdigest()
metadata["dataset_hash"] = hash_val

In [ ]:
import os
import json

base_path = f"{output_path}/{config['name']}"
os.makedirs(base_path, exist_ok=True)

df_systemic.to_parquet(f"{base_path}/systemic_features.parquet")
df_systemic_z.to_parquet(f"{base_path}/systemic_features_z.parquet")

with open(f"{base_path}/metadata.json", "w") as f:
    json.dump(metadata, f, indent=4)

## 7.1 Systemic Dataset export audit

In [ ]:
df_loaded = pd.read_parquet(f"{base_path}/systemic_features.parquet")
df_loaded_z = pd.read_parquet(f"{base_path}/systemic_features_z.parquet")

In [ ]:
assert df_loaded.equals(df_systemic), "Mismatch in systemic_features"
assert df_loaded_z.equals(df_systemic_z), "Mismatch in systemic_features_z"

In [ ]:
assert list(df_loaded.columns) == list(df_systemic.columns)
assert df_loaded.index.equals(df_systemic.index)

# 8. Visualization

In [ ]:
spy_price = panel["PRICE"]["SPY"]
spy_price = spy_price.loc[df_systemic.index]

In [ ]:
df_z = df_systemic_z

In [ ]:
spy_z = (spy_price - spy_price.mean()) / spy_price.std()

In [ ]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

features = df_z.columns
n = len(features)

fig = make_subplots(
    rows=n,
    cols=1,
    shared_xaxes=False,
    subplot_titles=[f"{col} vs SPY (Z)" for col in features]
)

for i, col in enumerate(features, start=1):
    
    # feature Z
    fig.add_trace(
        go.Scatter(
            x=df_z.index,
            y=df_z[col],
            name=col,
            line=dict(width=2)
        ),
        row=i, col=1
    )
    
    # SPY Z
    fig.add_trace(
        go.Scatter(
            x=spy_z.index,
            y=spy_z,
            name="SPY_Z",
            line=dict(dash="dot"),
            opacity=0.7
        ),
        row=i, col=1
    )

fig.update_layout(
    height=300 * n,
    title="Systemic Features (Z) vs SPY (Z)",
    showlegend=False,
    template = "plotly_dark"
)
fig.update_xaxes(
    dtick="M3",   # cada 3 meses
)

fig.show()